# QXYCell staged workflow

This notebook and the numbered Python scripts implement the **same staged workflow**. Use the scripts from a terminal or this notebook for an interactive, cell-by-cell experience.

For a first pass, run the cells in order. After stage 1, each stage reloads the saved checkpoint, so you can restart the kernel and rerun only a changed stage and its dependent stages.

## Configuration

Edit these values before running the workflow. Choose `"classifiers"` or `"table"` for `THRESHOLD_SOURCE`.

In [ ]:
from pathlib import Path

import qxycell as qxy


PROJECT_DIR = Path("/path/to/qupath_project")
OUTPUT_DIR = Path("/path/to/qxycell_output")
THRESHOLD_TABLE = PROJECT_DIR / "thresholds.tsv"
CELLTYPE_YAML = OUTPUT_DIR / "celltype" / "celltype_logic.yaml"
PIXEL_SIZE_UM = 0.28
IGNORE_ANNOTATION_TEXT = "ignore"
CELLTYPE_CONTEXT = (
    "Describe the tissue, disease, experimental groups, and expected cell "
    "populations here. The generated rules require expert review."
)
THRESHOLD_SOURCE = "classifiers"  # Choose "classifiers" or "table".

## Stage 1: import measurements

Create the base AnnData checkpoint. Normally rerun this stage only when the measurement files change.

In [ ]:
adata = qxy.import_cells(PROJECT_DIR, output_dir=OUTPUT_DIR)

## Stage 2: add annotations

Reload the checkpoint and add or replace current GeoJSON annotations and cell polygons.

In [ ]:
adata = qxy.load(OUTPUT_DIR)
qxy.add_annotations(
    adata,
    project_dir=PROJECT_DIR,
    pixel_size_um=PIXEL_SIZE_UM,
)

## Optional Stage 2b: remove ignored regions

Remove cells inside any annotation whose name contains `ignore`, matched case-insensitively. Use these annotations for tissue folds, damaged tissue, debris, edge artifacts, staining artifacts, or other excluded regions. This overwrites the shared checkpoint. If ignore polygons change later, rerun Stages 1, 2, and 2b in order before continuing.

In [ ]:
adata = qxy.load(OUTPUT_DIR)
qxy.remove_cells(
    adata,
    annotation_prefix="annotation__",
    remove_cells=IGNORE_ANNOTATION_TEXT,
    copy=False,
    verbose=True,
)
qxy.save(adata, output_dir=OUTPUT_DIR, verbose=True)

tables_dir = OUTPUT_DIR / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
adata.obs.to_csv(tables_dir / "cells_obs.csv")

## Stage 3: apply thresholds

The configuration selects exactly one source. Classifier mode ignores supplied tables and saves/replaces the applied values in `OUTPUT_DIR/thresholds/classifier_thresholds.tsv`; table mode uses only `THRESHOLD_TABLE` and never falls back to classifier JSON.

In [ ]:
adata = qxy.load(OUTPUT_DIR)

if THRESHOLD_SOURCE == "classifiers":
    qxy.threshold_from_classifiers(adata, project_dir=PROJECT_DIR)
elif THRESHOLD_SOURCE == "table":
    qxy.threshold_from_table(
        adata,
        THRESHOLD_TABLE,
        project_dir=PROJECT_DIR,
    )
else:
    raise ValueError(
        'THRESHOLD_SOURCE must be either "classifiers" or "table".'
    )

## Stage 4: generate the cell-type prompt

Generate a prompt using the thresholded marker names and your project context.

In [ ]:
adata = qxy.load(OUTPUT_DIR)
prompt = qxy.celltype_prompt(adata, context=CELLTYPE_CONTEXT)

## Pause for expert YAML review

Copy `OUTPUT_DIR/celltype/current_prompt.txt` into an LLM. Review and correct the returned YAML with an expert, then save it at `CELLTYPE_YAML`. Do not continue to stage 5 until that reviewed file exists.

## Stage 5: apply cell types

Reload the checkpoint and apply the reviewed cell-type YAML.

In [ ]:
adata = qxy.load(OUTPUT_DIR)
qxy.celltype(adata, CELLTYPE_YAML)

## Optional stage 6: plot spatial cell types

This is the same plot call as `06_plot_spatial_celltypes.py`. Every option is visible; `show=False` avoids interactive windows while PNG files are saved.

In [ ]:
adata = qxy.load(OUTPUT_DIR)
qxy.plot_spatial(
    adata,
    underlay_adata=None,
    category_col="celltype",
    sample_col=None,  # Prefer usable Sample labels, otherwise use Image.
    subset_col=None,
    subset_value=None,
    samples=None,
    celltypes=None,
    images=None,
    include_missing_samples=False,
    spatial_key=None,
    output_dir=None,
    filename_prefix=None,
    save_prefix=None,
    colors=None,
    palette=None,
    fixed_window_um=None,
    center_method="bbox",
    point_size=4.0,
    underlay_size=2.0,
    underlay_color="#bdbdbd",
    underlay_alpha=0.08,
    scale_bar=True,
    scale_bar_um=1000.0,
    scale_bar_label="1 mm",
    flip_y=True,
    figsize=(10.0, 10.0),
    auto_figsize=False,
    dpi=600,
    legend_width=2.2,
    combined=False,
    save_individual=True,
    save_png=True,
    save_pdf=False,
    max_cols=3,
    show=False,
    verbose=True,
)

## Rerun guide

| Changed input | Rerun |
|---|---|
| Measurement files | Stage 1, then all required later stages |
| Non-ignore annotation or cell GeoJSON | Stage 2, optionally stage 2b, stage 3, stage 4, stage 5, and optionally stage 6 |
| Ignore annotation polygons | Stages 1, 2, and 2b, then stages 3–5 and optionally stage 6 |
| Classifier JSON or threshold table | Stage 3, stage 4, stage 5, and optionally stage 6 |
| Prompt context | Stage 4 |
| Cell-type YAML | Stage 5, then optionally stage 6 |
| Spatial plot settings | Optional stage 6 |